# Sentence Share with the Definition Vector

| Item | Description |
|---|---|
| Goal | The sentence-level relevance score (1A) with c = the topic definition (T3.5.1): for each firm and sample, the share of its Item 1 sentences whose cosine similarity to c is at or above the threshold fixed in T3.6.5. |
| Input | `<scratch>/T3.6 - Firm Sentences/Processed/sentence_similarities.parquet` and `sentences.parquet`; `output/T3.4.1/filings.csv` (sample flags); `output/T3.6.5/similarity_threshold.csv` (the threshold for this vector). |
| Output | `output/T3.6.6/score_sentence_share_definition.csv`: one row per firm and sample with the counts and the score; `output/T3.6.6/example_sentences_above_threshold.csv`: 20 random sentences at or above the threshold. |
| Unit | A firm (gvkey) within a sample; the sentence is the unit the share is counted over. |
| Sample | All years (18,779 filings, 1,058 firms) and fiscal years 2010, 2015, 2020 (2,072 filings, 841 firms), as flagged in T3.4.1. |
| Variables | • `n_sentences`: all Item 1 sentences of the firm's filings in the sample.<br>• `n_above_threshold`: those with $\cos(e_k, c) \ge \tau$, where $\tau$ is read from `similarity_threshold.csv`.<br>• `score`: $\text{score}_{is} = n\_above\_threshold / n\_sentences$, numerator and denominator summed over the firm's filings in sample $s$. |

See `Topic Relevance Rebuild Plan.md` in the TEMP folder for the surrounding steps.


## Step 1. Import packages


In [1]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display


## Step 2. Define file locations and settings

The threshold is read from the file T3.6.5 saved, never retyped here.


In [2]:
PROJECT_FOLDER = Path("/project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis")
SCRATCH_FOLDER = Path("/scratch1/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3-TEMP")
NOTEBOOK_SUBFOLDER = Path("T3 - Topic Relevance and Corporate Disclosure/TEMP")
NOTEBOOK_NAME = "T3.6.6"  # the identifier names this notebook's output folder
PROCESSED_FOLDER = SCRATCH_FOLDER / "T3.6 - Firm Sentences" / "Processed"
SIMILARITIES_FILE = PROCESSED_FOLDER / "sentence_similarities.parquet"
SENTENCES_FILE = PROCESSED_FOLDER / "sentences.parquet"
FILINGS_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.4.1/filings.csv"
THRESHOLD_FILE = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output/T3.6.5/similarity_threshold.csv"
OUTPUT_FOLDER = PROJECT_FOLDER / NOTEBOOK_SUBFOLDER / "output" / NOTEBOOK_NAME

VECTOR = "definition"
SIMILARITY_COLUMN = "sim_definition"
SAMPLES = {"all_years": "in_all_years", "three_years": "in_three_years"}  # sample name to its flag in filings.csv
N_EXAMPLES = 20  # sentences shown from above the threshold
RANDOM_SEED = 0
OUTPUT_FOLDER.mkdir(parents=True, exist_ok=True)
print(f"Outputs: {OUTPUT_FOLDER}")


Outputs: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.6


## Step 3. Read the threshold

One row per vector in the T3.6.5 file; take this vector's threshold and the count it recorded, which the sentence table must reproduce.


In [3]:
threshold_table = pd.read_csv(THRESHOLD_FILE)
threshold_row = threshold_table.loc[threshold_table["vector"].eq(VECTOR)]
assert len(threshold_row) == 1
threshold_row = threshold_row.iloc[0]
SIMILARITY_THRESHOLD = float(threshold_row["threshold"])
print(f"Vector: {VECTOR}; threshold: {SIMILARITY_THRESHOLD}; decided by: {threshold_row['decided_by']}")
print(f"Reason: {threshold_row['reason']}")
print(f"Expected sentences at or above: {int(threshold_row['n_sentences_at_or_above']):,}")


Vector: definition; threshold: 0.67; decided by: author, by hand
Reason: Reviewed the example sentences by similarity band and the share tables; chose 0.67 for the definition vector and 0.75 for the centroid.
Expected sentences at or above: 6,183


## Step 4. Load the sentences with their similarities

The similarity table and the sentence table are in the same row order; join them on the row number.


In [4]:
similarities = pd.read_parquet(SIMILARITIES_FILE, columns=["row", SIMILARITY_COLUMN])
sentences = pd.read_parquet(SENTENCES_FILE)
assert len(similarities) == len(sentences)
assert (similarities["row"].to_numpy() == np.arange(len(sentences))).all()
sentences["similarity"] = similarities[SIMILARITY_COLUMN].to_numpy()
del similarities
print(f"Sentences: {len(sentences):,} from {sentences['accessionNo'].nunique():,} filings")
display(sentences.head(3))


Sentences: 5,647,505 from 18,779 filings


,accessionNo,gvkey,fyear,sent_idx,sentence,similarity
0,0000912057-99-011120,001013,1999,0,"ADC Telecommunications, Inc. (""ADC"") was incor...",0.366893
1,0000912057-99-011120,001013,1999,1,"ADC offers a broad range of network equipment,...",0.416137
2,0000912057-99-011120,001013,1999,2,"ADC's broadband, multiservice network solution...",0.434045


## Step 5. Flag the sentences at or above the threshold

The count must equal the one T3.6.5 recorded.


In [5]:
sentences["above_threshold"] = (sentences["similarity"] >= SIMILARITY_THRESHOLD).astype(int)
n_above = int(sentences["above_threshold"].sum())
assert n_above == int(threshold_row["n_sentences_at_or_above"])
print(f"Sentences at or above {SIMILARITY_THRESHOLD}: {n_above:,} ({n_above / len(sentences):.3%})")


Sentences at or above 0.67: 6,183 (0.109%)


## Step 6. Load the filings with their sample flags and firm names

Every sentence's filing must be in the T3.4.1 list.


In [6]:
filings = pd.read_csv(FILINGS_FILE, dtype={"gvkey": str, "cik": str})
assert filings["accessionNo"].is_unique
sentences = sentences.merge(
    filings[["accessionNo", "conm", "in_all_years", "in_three_years"]], on="accessionNo", how="left", validate="many_to_one"
)
assert sentences["in_all_years"].notna().all()
print(f"Filings: {len(filings):,}; sentences matched: {len(sentences):,}")


Filings: 18,779; sentences matched: 5,647,505


## Step 7. Show 20 sentences that passed the threshold

A random draw from the sentences at or above the threshold, with the firm, fiscal year, and similarity, so the reader sees what the score counts. The draw is saved beside the scores.


In [7]:
rng = np.random.default_rng(RANDOM_SEED)
above = sentences.loc[sentences["above_threshold"].eq(1)]
examples = above.iloc[np.sort(rng.choice(len(above), size=N_EXAMPLES, replace=False))]
examples = examples[["gvkey", "conm", "fyear", "accessionNo", "sent_idx", "similarity", "sentence"]].copy()
examples["similarity"] = examples["similarity"].round(3)
examples.to_csv(OUTPUT_FOLDER / "example_sentences_above_threshold.csv", index=False)
pd.set_option("display.max_colwidth", 220)
display(examples[["conm", "fyear", "similarity", "sentence"]].reset_index(drop=True))


,conm,fyear,similarity,sentence
0,AMERICAN INTERNATIONAL GROUP,2001,0.765,There is also legislation pending in the United States Congress and various states designed to provide additional privacy protections to consumer customers of financial institutions.
1,CIGNA GROUP (THE),2023,0.750,"Other federal and state laws that restrict the use and protect the privacy and security of PII exempt data and/or entities subject to HIPAA, but several states, such as, Nevada, and Connecticut, have recently enacted..."
2,COMCAST CORP,2018,0.725,"For example, California enacted the California Consumer Privacy Act (“CCPA”), which broadly regulates the sale of the consumer information of California residents and grants California residents certain rights to, am..."
3,US BANCORP,2019,0.799,"Data privacy and data protection are areas of increasing state legislative focus, and several states have recently enacted consumer privacy laws that impose compliance obligations with respect to personal information."
4,RAYMOND JAMES FINANCIAL INC,2022,0.714,Canadian data privacy laws contain many provisions similar to U.S. financial privacy laws and are currently undergoing legislative reform at a federal and provincial level.
5,UNITEDHEALTH GROUP INC,2025,0.765,Federal consumer protection laws may also apply in some instances to privacy and security practices related to personally identifiable information.
6,PAYPAL HOLDINGS INC,2023,0.677,Regulatory authorities are continuously considering numerous legislative and regulatory proposals and interpretive guidelines that may contain additional privacy and data protection obligations.
7,TIME WARNER INC,2002,0.684,"Certain areas of marketing activity are also subject to specific federal rules and statutes, such as the Telephone Consumer Protection Act, the Children’s Online Privacy Protection Act, the Gramm-Leach-Bliley Act (re..."
8,MATCH GROUP INC,2023,0.682,"Finally, talks of a U.S. federal privacy law are ongoing in Congress, with multiple proposals being considered, and may lead to the passing of a new law in the coming years."
9,TRADE DESK INC,2022,0.681,"In the United Kingdom and the European Union (including the European Economic Area (the “EEA”) and the countries of Iceland, Liechtenstein and Norway), or EU, separate laws and regulations (and member states’ impleme..."


## Step 8. Compute the score per firm and sample

For each sample, keep the firm's filings in that sample and sum the counts over them; the score is the ratio of the sums.


In [8]:
score_rows = []
for sample_name, flag_column in SAMPLES.items():
    in_sample = sentences.loc[sentences[flag_column].eq(1)]
    firm_counts = in_sample.groupby("gvkey").agg(
        conm=("conm", "first"),
        n_filings=("accessionNo", "nunique"),
        n_sentences=("sentence", "size"),
        n_above_threshold=("above_threshold", "sum"),
    ).reset_index()
    firm_counts["sample"] = sample_name
    firm_counts["score"] = firm_counts["n_above_threshold"] / firm_counts["n_sentences"]
    score_rows.append(firm_counts)
scores = pd.concat(score_rows, ignore_index=True)
scores["vector"] = VECTOR
scores["threshold"] = SIMILARITY_THRESHOLD
assert scores["score"].between(0, 1).all()
for sample_name in SAMPLES:
    part = scores.loc[scores["sample"].eq(sample_name)]
    print(f"{sample_name}: {len(part):,} firms, {int(part['n_filings'].sum()):,} filings; firms with a positive score: {int(part['score'].gt(0).sum()):,} ({part['score'].gt(0).mean():.1%}); median score among them {part.loc[part['score'].gt(0), 'score'].median():.4f}")
display(scores.head(3))


all_years: 1,058 firms, 18,779 filings; firms with a positive score: 306 (28.9%); median score among them 0.0021
three_years: 841 firms, 2,072 filings; firms with a positive score: 192 (22.8%); median score among them 0.0033


,gvkey,conm,n_filings,n_sentences,n_above_threshold,sample,score,vector,threshold
0,001013,ADC TELECOMMUNICATIONS INC,12,3786,0,all_years,0.0,definition,0.67
1,001045,AMERICAN AIRLINES GROUP INC,22,7288,0,all_years,0.0,definition,0.67
2,001075,PINNACLE WEST CAPITAL CORP,15,6823,0,all_years,0.0,definition,0.67


## Step 9. Show the ten firms with the highest score in each sample

A first look at who comes out on top; the figures in T3.9 do this properly.


In [9]:
for sample_name in SAMPLES:
    top = scores.loc[scores["sample"].eq(sample_name)].nlargest(10, "score")[["conm", "n_filings", "n_sentences", "n_above_threshold", "score"]]
    print(f"--- {sample_name}")
    display(top.reset_index(drop=True))


--- all_years


,conm,n_filings,n_sentences,n_above_threshold,score
0,EQUIFAX INC,27,7405,250,0.033761
1,TRADE DESK INC,10,2767,62,0.022407
2,APPLOVIN CORP,5,1114,23,0.020646
3,WESTERN UNION CO,20,7771,154,0.019817
4,META PLATFORMS INC,14,1870,37,0.019786
5,TWITTER INC,9,1419,28,0.019732
6,PAYPAL HOLDINGS INC,11,2368,46,0.019426
7,DUN & BRADSTREET HOLDNGS INC,5,1771,34,0.019198
8,OPTIMUM COMMUNICATIONS INC,9,4124,78,0.018914
9,BREAD FINANCIAL HOLDINGS INC,25,7244,134,0.018498


--- three_years


,conm,n_filings,n_sentences,n_above_threshold,score
0,EQUIFAX INC,3,981,36,0.036697
1,BREAD FINANCIAL HOLDINGS INC,3,641,21,0.032761
2,META PLATFORMS INC,2,183,5,0.027322
3,TRADE DESK INC,1,282,6,0.021277
4,TWITTER INC,2,289,6,0.020761
5,FOX CORP,1,307,6,0.019544
6,WESTERN UNION CO,3,1139,22,0.019315
7,TRIPADVISOR INC,2,479,9,0.018789
8,OPTIMUM COMMUNICATIONS INC,1,442,8,0.018100
9,PAYPAL HOLDINGS INC,2,474,8,0.016878


## Step 10. Save the scores

One row per firm and sample; the vector and threshold are repeated on every row so the file stands alone.


In [10]:
output_columns = ["gvkey", "conm", "sample", "vector", "threshold", "n_filings", "n_sentences", "n_above_threshold", "score"]
scores_file = OUTPUT_FOLDER / "score_sentence_share_definition.csv"
scores[output_columns].sort_values(["sample", "gvkey"]).to_csv(scores_file, index=False)
print(f"Saved: {scores_file}")
print(f"Rows: {len(scores):,}; columns: {output_columns}")


Saved: /project2/njia_1089/acroh/projects/LLM-Text-Analysis-and-Synthesis/T3 - Topic Relevance and Corporate Disclosure/TEMP/output/T3.6.6/score_sentence_share_definition.csv
Rows: 1,899; columns: ['gvkey', 'conm', 'sample', 'vector', 'threshold', 'n_filings', 'n_sentences', 'n_above_threshold', 'score']
